# S10 - Running LLMs Locally
## Exercises

These exercises cover running Large Language Models locally using three popular tools: **Ollama**, **LM Studio**, and **HuggingFace Transformers**. You will learn how to set up each tool, interact with local models programmatically, compare their approaches, and evaluate them on a real NLP task (sentiment analysis).

---

## Exercise 1: Text Generation with Ollama (Easy)

Ollama is a lightweight tool for running LLMs locally. It manages model downloads, quantization, and provides a simple CLI and REST API. Ollama runs models as a local server, exposing an OpenAI-compatible API on `http://localhost:11434`.

Before starting, make sure you have Ollama installed and running on your machine, with a small model pulled (e.g., `gemma3:1b` or `qwen2.5:1.5b`).

**Task:**
1. Use the `requests` library to send a POST request to Ollama's `/api/generate` endpoint
2. Send the prompt: `"Explain what sentiment analysis is in NLP in 2-3 sentences."`
3. Use a small model (e.g., `gemma3:1b`)
4. Set `stream: False` to get the complete response at once
5. Print the model's response text and the total generation time (available in the `total_duration` field of the response, in nanoseconds — convert to seconds)

In [3]:
import requests
import json

OLLAMA_URL = "http://localhost:11434/api/generate"

# Send a generation request to Ollama
# Model: "gemma3:1b" (or another small model you have pulled)
# Prompt: "Explain what sentiment analysis is in NLP in 2-3 sentences."
# Print the response and generation time

payload = {
    "model": "gemma3:1b",
    "prompt": "Explain what sentiment analysis is in NLP in 2-3 sentences.",
    "stream": False
}

response = requests.post(OLLAMA_URL, json=payload)
data = response.json()

print("Response:")
print(data["response"])
print(f"\nGeneration time: {data['total_duration'] / 1e9:.2f} seconds")

Response:
Sentiment analysis in Natural Language Processing (NLP) is the process of automatically determining the emotional tone or attitude expressed in a piece of text. It uses algorithms to identify words, phrases, and even entire sentences that convey positive, negative, or neutral feelings – essentially, it tries to understand *what* the text is saying.  Ultimately, it helps us gauge public opinion, track brand perception, and more.

Generation time: 8.96 seconds


---

## Exercise 2: Chat Completion with LM Studio (Easy)

LM Studio is a desktop application that lets you download, manage, and run LLMs locally with a graphical interface. Like Ollama, it exposes a local API server — but LM Studio uses the OpenAI-compatible chat completions format at `http://localhost:1234/v1`.

Before starting, make sure you have LM Studio installed, a small model loaded (e.g., a quantized Qwen, Gemma or Phi model), and the local server started.

**Task:**
1. Use the `openai` Python library to connect to LM Studio's local server
2. Set the `base_url` to `http://localhost:1234/v1` and use `api_key="lm-studio"` (any non-empty string works)
3. Send a chat completion request with a system message: `"You are an NLP expert."` and a user message: `"What are the main challenges in sentiment analysis?"`
4. Use the model identifier shown in LM Studio (you can also use `model="local-model"` as a generic identifier)
5. Print the response content and the token usage statistics (prompt tokens, completion tokens, total tokens)

In [4]:
from openai import OpenAI

# Connect to LM Studio's local server
# Send a chat completion request about sentiment analysis challenges
# Print the response and token usage

client = OpenAI(
    base_url="http://localhost:1234/v1",
    api_key="lm-studio"  # any non-empty string works for local servers
)

response = client.chat.completions.create(
    model="local-model",  # generic identifier, or paste the model name shown in LM Studio
    messages=[
        {"role": "system", "content": "You are an NLP expert."},
        {"role": "user", "content": "What are the main challenges in sentiment analysis?"}
    ]
)

print("Response:")
print(response.choices[0].message.content)

print("\nToken usage:")
print(f"  Prompt tokens     : {response.usage.prompt_tokens}")
print(f"  Completion tokens : {response.usage.completion_tokens}")
print(f"  Total tokens      : {response.usage.total_tokens}")

APIConnectionError: Connection error.

---

## Exercise 3: Local Inference with HuggingFace Transformers (Easy)

HuggingFace Transformers is the most widely used Python library for working with pre-trained models. Unlike Ollama and LM Studio (which manage model downloads and serve them via API), HuggingFace loads models directly into your Python process. The `pipeline` API provides a simple high-level interface for common NLP tasks.

**Task:**
1. Use `transformers.pipeline` to create a text generation pipeline with a small model (e.g., `"TinyLlama/TinyLlama-1.1B-Chat-v1.0"` or `"Qwen/Qwen2.5-0.5B-Instruct"`)
2. Send the prompt: `"Sentiment analysis is important because"`
3. Set `max_new_tokens=100` and `do_sample=True` with `temperature=0.7`
4. Print the generated text
5. Measure and print the inference time using `time.time()`

In [ ]:
from transformers import pipeline
import time

# Create a text generation pipeline with a small model
# Generate text and measure inference time

generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",  # ~1GB, fast to download and run on CPU
)

prompt = "Sentiment analysis is important because"

start = time.time()

output = generator(
    prompt,
    max_new_tokens=100,
    do_sample=True,
    temperature=0.7,
)

elapsed = time.time() - start

print("Generated text:")
print(output[0]["generated_text"])
print(f"\nInference time: {elapsed:.2f} seconds")

c:\Users\carid\Desktop\UIE\3º\SEGUNDO CUATRIMESTRE\Procesamiento del Lenguaje\procesamiento\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\carid\Desktop\UIE\3º\SEGUNDO CUATRIMESTRE\Procesamiento del Lenguaje\procesamiento\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\carid\.cache\huggingface\hub\models--Qwen--Qwen2.5-0.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks o

---

## Exercise 4: Comparing the Three Tools (Medium)

Now that you have used all three tools, it is time to compare them systematically. Each tool has different strengths: Ollama is optimized for ease of use via CLI/API, LM Studio provides a GUI with fine-grained control, and HuggingFace offers the deepest integration with the Python ecosystem.

**Task:**
1. Send the **same prompt** to all three tools: `"Classify the sentiment of the following text as Positive, Negative, or Neutral. Only respond with the sentiment label.\n\nText: 'The movie was absolutely wonderful, I loved every minute of it'\n\nSentiment:"`
2. For each tool, record:
   - The response text
   - The approximate response time (use `time.time()` before and after each call)
   - Any qualitative observations (response quality, formatting, etc.)
3. Print a summary table comparing the three tools
4. Write a brief paragraph (as a markdown cell or as a Python comment) explaining which tool you found easiest to use and why

In [ ]:
import time
import requests
from openai import OpenAI

results = {}

# --- Ollama ---
# Send the sentiment classification prompt and record response + time

# --- LM Studio ---
# Send the same prompt and record response + time

# --- HuggingFace ---
# Send the same prompt and record response + time

# Print comparison table

**Your comparison:** *(Write your observations here: Which tool did you find easiest to use? Which gave the best response? What are the trade-offs?)*

---

## Exercise 5: Evaluating Local LLMs on Sentiment Analysis (Hard)

In this exercise, you will design and run a proper evaluation of the three tools on a sentiment analysis task. You will use a small labeled dataset, send each sample to each tool, and compute standard classification metrics.

The goal is not just to measure accuracy, but to understand how different local LLM setups handle a real NLP task — including response parsing, consistency, and failure modes.

**Task:**
1. Create a test dataset with at least 10 labeled sentences covering Positive, Negative, and Neutral sentiments (or use a subset from an existing dataset like SST-2 or the financial tweets dataset from the previous lab)
2. Write a function for each tool that takes a text and returns the predicted sentiment label (`"Positive"`, `"Negative"`, or `"Neutral"`)
3. Run all test sentences through each tool
4. For each tool, compute and print:
   - Accuracy
   - Per-class precision, recall, and F1 (use `sklearn.metrics.classification_report`)
   - A confusion matrix
5. Write a final analysis (as markdown or comments) discussing:
   - Which tool performed best on this task and why
   - What failure modes did you observe (e.g., verbose responses, refusal to classify, inconsistent formatting)
   - How you handled parsing the model's response into a clean label
   - Your recommendation for which tool to use for local NLP tasks

In [ ]:
import time
import requests
import numpy as np
from openai import OpenAI
from sklearn.metrics import classification_report, confusion_matrix

# --- Test Dataset ---
test_data = [
    # Add at least 10 labeled (text, label) pairs
    # ("text here", "Positive"),
    # ("text here", "Negative"),
    # ("text here", "Neutral"),
]

# --- Prediction Functions ---
def predict_ollama(text):
    """Send text to Ollama and return predicted sentiment label."""
    pass

def predict_lmstudio(text):
    """Send text to LM Studio and return predicted sentiment label."""
    pass

def predict_huggingface(text):
    """Send text to HuggingFace pipeline and return predicted sentiment label."""
    pass

# --- Run Evaluation ---
# For each tool, predict sentiment for all test sentences
# Compute and print classification_report and confusion_matrix

**Your evaluation analysis:** *(Discuss your results here: Which tool performed best? What failure modes did you observe? How did you handle response parsing? What is your recommendation?)*